# 02 — WordNet Phrase Construction

**Primary author:** Victoria

**Builds on:**
- *02_embedding_generation.ipynb* (Victoria — phrase construction logic, CALE delimiter insertion, WordNet synset lookup)
- *01_wn_filtering_and_split.ipynb* (Victoria — produces filtered clue file and vocabulary)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

---

Before a word can be embedded by CALE, it must be placed inside a tagged
passage of text — this is the **f** (phrase construction) step in the
word → f(word) → g(f(word)) → embedding chain. Different f strategies
encode different assumptions about which sense of a word to capture and how
much context to include. This notebook constructs phrases for the three
WordNet-based f strategies:

- **f_clue** — wraps the definition word(s) in `<t></t>` delimiters within
  the full clue surface text, capturing how the clue's misleading context
  shifts the definition's meaning. This is the "treatment" condition in our
  ATE framework.
- **f_common_wndef** — builds a decontextualized phrase from the word's
  WordNet definition (`"<t>word</t>: definition"`). This provides a clean
  sense-specific baseline without misleading clue context — used as the T=0
  control in ATE evaluation and as positive/negative examples in triplet
  training.
- **f_common_wnex** — embeds the word inside a WordNet usage example
  sentence, providing natural-language context without the dictionary-entry
  format of wndef. This lets us test whether g_1 generalizes beyond the
  specific format it was trained on (the formatting hypothesis).

For each vocabulary-based f (wndef, wnex), the notebook also produces the
subset-specific vocabulary files, filtered clue files, and validation
vocabulary files that define that f's data scope. Each f is strictly defined:
if a word lacks the required WordNet resource, it is absent from that f —
no fallbacks.

Future resource families (dictionary APIs, LLM-generated phrases) will have
their own `02_phrase_construction_<resource>.ipynb` notebooks.

**Reads:**
- `data/filtered_split/wn_synset/clues_wn_filtered.csv` (239,406 rows)
- `data/filtered_split/wn_synset/vocabulary.csv` (53,930 words)
- `data/filtered_split/wn_synset/vocabulary_val.csv` (26,152 words)
- `../../notebooks/clue_utils.py` — shared definition-finding and delimiter-placement logic
- WordNet corpus via NLTK

**Writes to:** `data/filtered_split/wn_synset/`
- `clue_phrases/f_clue.csv` — one row per (clue_id, definition) with valid tagging
- `wndef/f_common_wndef.csv` — one row per word with WN definition phrase
- `wndef/vocabulary_wndef.csv` — words with valid f_common_wndef phrase
- `wndef/vocabulary_wndef_val.csv` — validation-split subset
- `wndef/clues_wndef_filtered.csv` — clue rows where both def and ans have wndef phrases
- `wnex/f_common_wnex.csv` — one row per word with WN usage example phrase
- `wnex/vocabulary_wnex.csv` — words with valid f_common_wnex phrase
- `wnex/vocabulary_wnex_val.csv` — validation-split subset
- `wnex/clues_wnex_filtered.csv` — clue rows where both def and ans have wnex phrases

---

## §1 — Imports and Configuration

Environment auto-detection lets this notebook run unmodified on Local, Great
Lakes, and Colab. We import `tag_definition_in_surface` from the shared
`clue_utils.py` module — the same function used in
`structural_filtering.ipynb` — ensuring that delimiter placement in f_clue
phrases is always consistent with the definition-in-surface matching used
to build `clues_filtered.csv`.

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================
import time
import sys
import re
import pandas as pd
import numpy as np
from pathlib import Path

import nltk
from nltk.corpus import wordnet as wn

try:
    wn.synsets("test")
except LookupError:
    nltk.download("wordnet", quiet=True)

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    PROJECT_ROOT = Path("../..").resolve()

COMPONENT_ROOT = PROJECT_ROOT / "custom_embedding_model"
DATA_DIR       = COMPONENT_ROOT / "data" / "filtered_split" / "wn_synset"
OUTPUT_DIR     = COMPONENT_ROOT / "outputs"

CLUE_PHRASES_DIR = DATA_DIR / "clue_phrases"
WNDEF_DIR        = DATA_DIR / "wndef"
WNEX_DIR         = DATA_DIR / "wnex"

for d in [CLUE_PHRASES_DIR, WNDEF_DIR, WNEX_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
from clue_utils import tag_definition_in_surface  # canonical tagger shared with structural_filtering

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment: {env_label}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR:     {DATA_DIR}")

# Version reporting (Decision 18)
print(f"\npandas:          {pd.__version__}")
print(f"numpy:           {np.__version__}")
print(f"nltk:            {nltk.__version__}")
print(f"WordNet corpus:  {wn.get_version()}")

In [ ]:
# ============================================================
# Load input data
# ============================================================
t0 = time.time()

df = pd.read_csv(
    DATA_DIR / "clues_wn_filtered.csv",
    keep_default_na=False, na_values=[""],  # "nan" is a valid crossword word
)
vocab = pd.read_csv(
    DATA_DIR / "vocabulary.csv",
    keep_default_na=False, na_values=[""],
)
vocab_val = pd.read_csv(
    DATA_DIR / "vocabulary_val.csv",
    keep_default_na=False, na_values=[""],
)

assert len(df) == 239_406, f"Expected 239,406 rows, got {len(df):,}"
assert len(vocab) == 53_930, f"Expected 53,930 words, got {len(vocab):,}"

print(f"Loaded clues_wn_filtered.csv: {len(df):,} rows")
print(f"Loaded vocabulary.csv:        {len(vocab):,} words")
print(f"Loaded vocabulary_val.csv:    {len(vocab_val):,} words")
df.head(3)

print(f"\nExample rows from clues_wn_filtered.csv:")
for _, r in df.head(3).iterrows():
    print(f"  surface='{r['surface'][:60]}...' definition='{r['definition']}' answer='{r['answer']}'")


---

## §2 — f_clue Construction

f_clue is the "treatment" phrase construction strategy: it takes a clue's
surface text and wraps the definition word(s) in `<t></t>` delimiters. CALE's
architecture focuses the resulting embedding on the delimited span, capturing
how the clue's misleading surface context shifts the definition's perceived
meaning. The embedding g(f_clue(definition)) is what we compare against the
decontextualized baseline g(f(word)) to measure the Average Treatment Effect.

`tag_definition_in_surface` returns `None` when the definition cannot be
unambiguously located (e.g., the definition string appears more than once
in the surface, making delimiter placement ambiguous). These rows are dropped
from `f_clue.csv` but remain in `clues_wn_filtered.csv` — f_clue does not
further filter the upstream scope.

In [ ]:
# ============================================================
# Construct f_clue phrases
# ============================================================
t_clue_start = time.time()

phrases = []
failures = []

for _, row in df.iterrows():
    tagged = tag_definition_in_surface(row["definition"], row["surface"])  # wraps definition in <t></t> for CALE
    if tagged is not None:  # None when definition can't be unambiguously located
        phrases.append({
            "clue_id": row["clue_id"],
            "definition": row["definition"],
            "split": row["split"],
            "phrase": tagged,
        })
    else:
        failures.append({
            "clue_id": row["clue_id"],
            "definition": row["definition"],
            "surface": row["surface"],
        })

df_fclue = pd.DataFrame(phrases)
df_failures = pd.DataFrame(failures)

t_clue_elapsed = time.time() - t_clue_start
print(f"f_clue construction: {t_clue_elapsed:.1f}s")
print(f"Valid phrases:  {len(df_fclue):,} / {len(df):,} ({len(df_fclue)/len(df):.1%})")
print(f"Failures:       {len(df_failures):,} ({len(df_failures)/len(df):.1%})")

print(f"\nExample f_clue phrases:")
for _, r in df_fclue.head(5).iterrows():
    print(f"  {r['phrase'][:100]}{'...' if len(r['phrase']) > 100 else ''}")


In [ ]:
# ============================================================
# Validate and save f_clue.csv
# ============================================================
assert df_fclue.duplicated(subset=["clue_id", "definition"]).sum() == 0, \
    "Duplicate (clue_id, definition) pairs found in f_clue"
assert (df_fclue["phrase"].str.count("<t>") == 1).all(), "Some phrases missing or have extra <t>"
assert (df_fclue["phrase"].str.count("</t>") == 1).all(), "Some phrases missing or have extra </t>"

# Rejoin source to verify split labels survived construction unchanged
fclue_merged = df_fclue.merge(
    df[["clue_id", "definition", "split"]],
    on=["clue_id", "definition"],
    suffixes=("", "_source"),
)
assert (fclue_merged["split"] == fclue_merged["split_source"]).all(), \
    "Split mismatch between f_clue and source data"

df_fclue.to_csv(CLUE_PHRASES_DIR / "f_clue.csv", index=False)
print(f"All validations passed.")
print(f"Saved f_clue.csv: {len(df_fclue):,} rows")
print(f"  Location: {(CLUE_PHRASES_DIR / 'f_clue.csv').relative_to(COMPONENT_ROOT)}")

if len(df_failures) > 0:
    print(f"\nSample failures (first 5):")
    for _, r in df_failures.head(5).iterrows():
        print(f"  clue_id={r['clue_id']}, definition='{r['definition']}', surface='{r['surface']}'")

if len(df_failures) > 0:
    print(f"\n(Failures typically occur when the definition string appears more than once")
    print(f" in the surface text, making <t></t> delimiter placement ambiguous.)")


---

## §3 — f_common_wndef Construction

f_common_wndef provides the decontextualized baseline: for each word, we
construct `"<t>word</t>: <WordNet definition>"` using the most frequent
synset (index 0). Because CALE focuses its embedding on the `<t></t>`-delimited
span, this produces an embedding of the word in the context of its own
dictionary definition — a clean sense signal without any misleading clue
context. These phrases serve as the T=0 baseline in ATE evaluation and as
positive/negative examples in triplet training.

Every word in `vocabulary.csv` has at least one synset (guaranteed by the
NB 01 filter), so f_common_wndef should cover the entire vocabulary.

A `self_ref` boolean flag marks words whose own name appears in their WordNet
definition (e.g., "plant" defined as "buildings for carrying on industrial
labor"). These are kept — not filtered out — but the flag lets downstream
evaluation check whether self-referential definitions produce systematically
different embedding behavior.

In [ ]:
# ============================================================
# Construct f_common_wndef phrases
# ============================================================
t_wndef_start = time.time()

wndef_records = []
wndef_warnings = []

for _, vrow in vocab.iterrows():
    word = vrow["word"]
    synsets = wn.synsets(word)
    if not synsets:
        wndef_warnings.append(word)
        continue
    synset = synsets[0]  # use the most common sense (WordNet convention)
    display_word = word.replace("_", " ")  # undo WordNet underscore convention for natural text
    phrase = f"<t>{display_word}</t>: {synset.definition()}"  # CALE format: <t>target</t> followed by context

    # Detect self-reference: does the target word appear untagged in the phrase?
    remainder = phrase.replace(f"<t>{display_word}</t>", "", 1)  # strip the tagged occurrence so we only search the definition text
    # Flag words that appear in their own WN definition (e.g., "plant" in "plant life")
    self_ref = bool(re.search(
        r"\b" + re.escape(display_word) + r"\b", remainder, re.IGNORECASE
    ))

    wndef_records.append({
        "word": word,
        "synset_name": synset.name(),
        "phrase": phrase,
        "self_ref": self_ref,
    })

if wndef_warnings:
    print(f"WARNING: {len(wndef_warnings)} words with no synsets: {wndef_warnings[:10]}")
else:
    print("All vocabulary words have synsets (as expected).")

df_wndef = pd.DataFrame(wndef_records)
df_wndef = df_wndef.sort_values("word").reset_index(drop=True)  # canonical alphabetical ordering for embedding alignment
df_wndef["row"] = range(len(df_wndef))

t_wndef_elapsed = time.time() - t_wndef_start
n_self_ref = df_wndef["self_ref"].sum()
print(f"f_common_wndef construction: {t_wndef_elapsed:.1f}s")
print(f"Words with valid phrase: {len(df_wndef):,} / {len(vocab):,} ({len(df_wndef)/len(vocab):.1%})")
print(f"Self-referential phrases: {n_self_ref:,} ({n_self_ref/len(df_wndef):.1%})")

print(f"\nExample f_common_wndef phrases:")
for _, r in df_wndef.head(5).iterrows():
    print(f"  [{r['synset_name']}] {r['phrase'][:90]}{'...' if len(r['phrase']) > 90 else ''} (self_ref={r['self_ref']})")

self_ref_examples = df_wndef[df_wndef['self_ref']].head(3)
if len(self_ref_examples) > 0:
    print(f"\nExample self-referential phrases:")
    for _, r in self_ref_examples.iterrows():
        print(f"  [{r['synset_name']}] {r['phrase'][:90]}{'...' if len(r['phrase']) > 90 else ''}")


In [ ]:
# ============================================================
# Validate and save f_common_wndef.csv
# ============================================================
assert (df_wndef["phrase"].str.count("<t>") == 1).all(), "Some wndef phrases missing or have extra <t>"
assert (df_wndef["phrase"].str.count("</t>") == 1).all(), "Some wndef phrases missing or have extra </t>"
assert df_wndef["word"].duplicated().sum() == 0, "Duplicate words in f_common_wndef"
assert df_wndef["self_ref"].dtype == bool, f"self_ref dtype is {df_wndef['self_ref'].dtype}, expected bool"
assert len(df_wndef) == len(vocab) or len(wndef_warnings) > 0, \
    "Unexpected word count mismatch without warnings"

df_wndef[["word", "row", "synset_name", "phrase", "self_ref"]].to_csv(
    WNDEF_DIR / "f_common_wndef.csv", index=False
)
print(f"All validations passed.")
print(f"Saved f_common_wndef.csv: {len(df_wndef):,} rows")
print(f"  Location: {(WNDEF_DIR / 'f_common_wndef.csv').relative_to(COMPONENT_ROOT)}")

---

## §4 — f_common_wndef Vocabulary and Filtered Clues

Each f strategy defines its own data scope. A clue row is usable for wndef
evaluation only if *both* its definition and answer have valid f_common_wndef
phrases — otherwise we cannot compute the T=0 similarity that the ATE
requires. This section builds three artifacts that define the wndef scope:

1. **vocabulary_wndef.csv** — the set of words with valid phrases, with its
   own canonical row ordering (the index for wndef embedding arrays).
2. **clues_wndef_filtered.csv** — clue rows where both definition_wn and
   answer_wn appear in the wndef vocabulary.
3. **vocabulary_wndef_val.csv** — the subset of wndef words that appear in
   validation-split rows of the *filtered* clue file, ensuring we generate
   only the embeddings needed for ATE evaluation.

In [ ]:
# ============================================================
# Build wndef vocabulary and filtered clues
# ============================================================
wndef_words = set(df_wndef["word"])

# --- vocabulary_wndef.csv ---
vocab_wndef = pd.DataFrame({"word": sorted(wndef_words)})
vocab_wndef["row"] = range(len(vocab_wndef))
vocab_wndef.to_csv(WNDEF_DIR / "vocabulary_wndef.csv", index=False)
print(f"vocabulary_wndef.csv: {len(vocab_wndef):,} words ({len(vocab_wndef)/len(vocab):.1%} of full vocabulary)")

# --- clues_wndef_filtered.csv ---
df_wndef_filt = df[
    df["definition_wn"].isin(wndef_words) & df["answer_wn"].isin(wndef_words)  # both sides must have phrases
].copy()
df_wndef_filt.to_csv(WNDEF_DIR / "clues_wndef_filtered.csv", index=False)
print(f"clues_wndef_filtered.csv: {len(df_wndef_filt):,} rows ({len(df_wndef_filt)/len(df):.1%} of clues_wn_filtered)")

for split_name in ["train", "validate", "test"]:
    n = (df_wndef_filt["split"] == split_name).sum()
    print(f"  {split_name}: {n:,} ({n/len(df_wndef_filt):.1%})")

# --- vocabulary_wndef_val.csv ---
wndef_val_rows = df_wndef_filt[df_wndef_filt["split"] == "validate"]
wndef_val_words = sorted(
    (set(wndef_val_rows["definition_wn"]) | set(wndef_val_rows["answer_wn"])) & wndef_words  # union of def+ans words, intersected with wndef scope
)
vocab_wndef_val = pd.DataFrame({"word": wndef_val_words})
vocab_wndef_val["row"] = range(len(vocab_wndef_val))
vocab_wndef_val.to_csv(WNDEF_DIR / "vocabulary_wndef_val.csv", index=False)
print(f"vocabulary_wndef_val.csv: {len(vocab_wndef_val):,} words ({len(vocab_wndef_val)/len(vocab_wndef):.1%} of wndef vocabulary)")

In [ ]:
# ============================================================
# Validate wndef artifacts
# ============================================================
assert set(vocab_wndef_val["word"]).issubset(set(vocab_wndef["word"])), \
    "Val vocabulary contains words not in full wndef vocabulary"
assert df_wndef_filt["definition_wn"].isin(wndef_words).all(), \
    "Some definitions in filtered clues not in wndef vocabulary"
assert df_wndef_filt["answer_wn"].isin(wndef_words).all(), \
    "Some answers in filtered clues not in wndef vocabulary"

# Splits are assigned at the (definition, answer) pair level; verify no pair spans two splits
pair_splits = df_wndef_filt.groupby(["definition", "answer"])["split"].nunique()
assert (pair_splits == 1).all(), \
    "Some (definition, answer) pairs span multiple splits"

print("All wndef validations passed.")

---

## §5 — f_common_wnex Construction

f_common_wnex embeds the word inside a natural-language usage example from
WordNet, with the target word wrapped in `<t></t>` delimiters. Unlike
f_common_wndef's dictionary-entry format, these phrases resemble ordinary
sentences — making wnex a critical tool for testing the formatting hypothesis:
if g_1 (trained on wndef-format phrases) also changes the embedding behavior
of wnex-format phrases, that suggests semantic generalization rather than
format-specific overfitting.

A word is excluded from f_common_wnex if its most-frequent synset has no
usage examples, or if none of the examples contain the word exactly once
(case-insensitive word-boundary match). Coverage is expected to be
substantially lower than wndef, since many WordNet synsets lack examples
entirely.

In [ ]:
# ============================================================
# Construct f_common_wnex phrases
# ============================================================
t_wnex_start = time.time()

wnex_records = []
wnex_no_examples = 0
wnex_no_single_match = 0

for _, vrow in vocab.iterrows():
    word = vrow["word"]
    synsets = wn.synsets(word)
    if not synsets:
        continue
    synset = synsets[0]  # most common sense
    examples = synset.examples()
    if not examples:
        wnex_no_examples += 1
        continue

    display_word = word.replace("_", " ")
    pattern = re.compile(r"\b" + re.escape(display_word) + r"\b", re.IGNORECASE)

    found = False
    for example in examples:
        matches = list(pattern.finditer(example))
        if len(matches) == 1:  # require exactly one occurrence for unambiguous <t></t> placement
            m = matches[0]
            phrase = example[:m.start()] + "<t>" + example[m.start():m.end()] + "</t>" + example[m.end():]  # wrap the single match in CALE delimiters
            wnex_records.append({
                "word": word,
                "synset_name": synset.name(),
                "phrase": phrase,
            })
            found = True  # accept the first example with a single match
            break

    if not found:
        wnex_no_single_match += 1

df_wnex = pd.DataFrame(wnex_records)
df_wnex = df_wnex.sort_values("word").reset_index(drop=True)
df_wnex["row"] = range(len(df_wnex))

t_wnex_elapsed = time.time() - t_wnex_start
print(f"f_common_wnex construction: {t_wnex_elapsed:.1f}s")
print(f"Words with valid phrase: {len(df_wnex):,} / {len(vocab):,} ({len(df_wnex)/len(vocab):.1%})")
print(f"  No examples in synset[0]: {wnex_no_examples:,}")
print(f"  Examples exist but no single match: {wnex_no_single_match:,}")

print(f"\nCoverage is expected to be substantially lower than wndef because many synsets lack examples.")

print(f"\nExample f_common_wnex phrases:")
for _, r in df_wnex.head(5).iterrows():
    print(f"  [{r['synset_name']}] {r['phrase'][:100]}{'...' if len(r['phrase']) > 100 else ''}")


In [ ]:
# ============================================================
# Validate and save f_common_wnex.csv
# ============================================================
assert (df_wnex["phrase"].str.count("<t>") == 1).all(), "Some wnex phrases missing or have extra <t>"
assert (df_wnex["phrase"].str.count("</t>") == 1).all(), "Some wnex phrases missing or have extra </t>"
assert df_wnex["word"].duplicated().sum() == 0, "Duplicate words in f_common_wnex"

df_wnex[["word", "row", "synset_name", "phrase"]].to_csv(
    WNEX_DIR / "f_common_wnex.csv", index=False
)
print(f"All validations passed.")
print(f"Saved f_common_wnex.csv: {len(df_wnex):,} rows")
print(f"  Location: {(WNEX_DIR / 'f_common_wnex.csv').relative_to(COMPONENT_ROOT)}")

---

## §6 — f_common_wnex Vocabulary and Filtered Clues

Same three-artifact structure as §4 — vocabulary, filtered clues, and
validation vocabulary — but for the wnex scope. Because fewer words have
valid wnex phrases, we expect a substantially smaller filtered clue set.
The row loss compounds: a clue is retained only if *both* its definition
and answer have wnex phrases, so even a modest vocabulary reduction can
exclude a large fraction of clue rows.

In [ ]:
# ============================================================
# Build wnex vocabulary and filtered clues
# ============================================================
wnex_words = set(df_wnex["word"])

# --- vocabulary_wnex.csv ---
vocab_wnex = pd.DataFrame({"word": sorted(wnex_words)})
vocab_wnex["row"] = range(len(vocab_wnex))
vocab_wnex.to_csv(WNEX_DIR / "vocabulary_wnex.csv", index=False)
print(f"vocabulary_wnex.csv: {len(vocab_wnex):,} words ({len(vocab_wnex)/len(vocab):.1%} of full vocabulary)")

# --- clues_wnex_filtered.csv ---
df_wnex_filt = df[
    df["definition_wn"].isin(wnex_words) & df["answer_wn"].isin(wnex_words)  # both sides must have phrases
].copy()
df_wnex_filt.to_csv(WNEX_DIR / "clues_wnex_filtered.csv", index=False)
print(f"clues_wnex_filtered.csv: {len(df_wnex_filt):,} rows ({len(df_wnex_filt)/len(df):.1%} of clues_wn_filtered)")

for split_name in ["train", "validate", "test"]:
    n = (df_wnex_filt["split"] == split_name).sum()
    print(f"  {split_name}: {n:,} ({n/len(df_wnex_filt):.1%})")

# --- vocabulary_wnex_val.csv ---
wnex_val_rows = df_wnex_filt[df_wnex_filt["split"] == "validate"]
wnex_val_words = sorted(
    (set(wnex_val_rows["definition_wn"]) | set(wnex_val_rows["answer_wn"])) & wnex_words  # union of def+ans words, intersected with wnex scope
)
vocab_wnex_val = pd.DataFrame({"word": wnex_val_words})
vocab_wnex_val["row"] = range(len(vocab_wnex_val))
vocab_wnex_val.to_csv(WNEX_DIR / "vocabulary_wnex_val.csv", index=False)
print(f"vocabulary_wnex_val.csv: {len(vocab_wnex_val):,} words ({len(vocab_wnex_val)/len(vocab_wnex):.1%} of wnex vocabulary)")

In [ ]:
# ============================================================
# Validate wnex artifacts
# ============================================================
assert set(vocab_wnex_val["word"]).issubset(set(vocab_wnex["word"])), \
    "Val vocabulary contains words not in full wnex vocabulary"
assert df_wnex_filt["definition_wn"].isin(wnex_words).all(), \
    "Some definitions in filtered clues not in wnex vocabulary"
assert df_wnex_filt["answer_wn"].isin(wnex_words).all(), \
    "Some answers in filtered clues not in wnex vocabulary"

# Splits are assigned at the (definition, answer) pair level; verify no pair spans two splits
pair_splits = df_wnex_filt.groupby(["definition", "answer"])["split"].nunique()
assert (pair_splits == 1).all(), \
    "Some (definition, answer) pairs span multiple splits"

print("All wnex validations passed.")

---

## §7 — Summary Statistics and Results File

Compile coverage statistics across all three f strategies and write the
results file. The cross-f comparison quantifies the vocabulary and clue
overlap between wndef and wnex — important because the formatting hypothesis
test (Stage 5, Step B) requires words that appear in *both* vocabularies to
compare g_1's behavior across phrase formats.

In [ ]:
# ============================================================
# Summary statistics and results file
# ============================================================
elapsed_total = time.time() - t0

# Cross-f comparison
wndef_only = wndef_words - wnex_words
wnex_only  = wnex_words - wndef_words
both_words = wndef_words & wnex_words

# Compare clue coverage between the two resource families at the row level
wndef_filt_keys = set(zip(df_wndef_filt["clue_id"], df_wndef_filt["definition"]))
wnex_filt_keys  = set(zip(df_wnex_filt["clue_id"], df_wnex_filt["definition"]))
rows_wndef_not_wnex = len(wndef_filt_keys - wnex_filt_keys)

# Split fractions
wndef_splits = {s: (df_wndef_filt["split"] == s).sum() for s in ["train", "validate", "test"]}
wnex_splits  = {s: (df_wnex_filt["split"] == s).sum() for s in ["train", "validate", "test"]}

n_self_ref = df_wndef["self_ref"].sum()

lines_out = [
    "# Results: 02 — WordNet Phrase Construction\n",
    "",
    "## Versions\n",
    f"- pandas: {pd.__version__}",
    f"- numpy: {np.__version__}",
    f"- nltk: {nltk.__version__}",
    f"- WordNet corpus: {wn.get_version()}",
    "",
    "## f_clue Coverage\n",
    f"- Input rows (`clues_wn_filtered.csv`): {len(df):,}",
    f"- Rows with valid f_clue phrase: {len(df_fclue):,} ({len(df_fclue)/len(df):.1%})",
    f"- Rows dropped: {len(df_failures):,} ({len(df_failures)/len(df):.1%})",
    "",
    "## f_common_wndef Coverage\n",
    f"- Full vocabulary size: {len(vocab):,}",
    f"- Words with valid phrase: {len(df_wndef):,} ({len(df_wndef)/len(vocab):.1%})",
    f"- Self-referential phrases: {n_self_ref:,} ({n_self_ref/len(df_wndef):.1%})",
    f"- `clues_wndef_filtered.csv` rows: {len(df_wndef_filt):,} ({len(df_wndef_filt)/len(df):.1%} of clues_wn_filtered)",
    f"- `vocabulary_wndef.csv`: {len(vocab_wndef):,} words",
    f"- `vocabulary_wndef_val.csv`: {len(vocab_wndef_val):,} words",
    f"- Split fractions in clues_wndef_filtered:",
    f"  - Train: {wndef_splits['train']:,} ({wndef_splits['train']/len(df_wndef_filt):.1%})",
    f"  - Validate: {wndef_splits['validate']:,} ({wndef_splits['validate']/len(df_wndef_filt):.1%})",
    f"  - Test: {wndef_splits['test']:,} ({wndef_splits['test']/len(df_wndef_filt):.1%})",
    "",
    "## f_common_wnex Coverage\n",
    f"- Full vocabulary size: {len(vocab):,}",
    f"- Words with valid phrase: {len(df_wnex):,} ({len(df_wnex)/len(vocab):.1%})",
    f"- `clues_wnex_filtered.csv` rows: {len(df_wnex_filt):,} ({len(df_wnex_filt)/len(df):.1%} of clues_wn_filtered)",
    f"- `vocabulary_wnex.csv`: {len(vocab_wnex):,} words",
    f"- `vocabulary_wnex_val.csv`: {len(vocab_wnex_val):,} words",
    f"- Split fractions in clues_wnex_filtered:",
    f"  - Train: {wnex_splits['train']:,} ({wnex_splits['train']/len(df_wnex_filt):.1%})",
    f"  - Validate: {wnex_splits['validate']:,} ({wnex_splits['validate']/len(df_wnex_filt):.1%})",
    f"  - Test: {wnex_splits['test']:,} ({wnex_splits['test']/len(df_wnex_filt):.1%})",
    "",
    "## Cross-f Comparison\n",
    f"- Words in wndef but not wnex: {len(wndef_only):,}",
    f"- Words in wnex but not wndef: {len(wnex_only):,}",
    f"- Words in both: {len(both_words):,}",
    f"- Rows in clues_wndef_filtered but not clues_wnex_filtered: {rows_wndef_not_wnex:,}",
    "",
    "## Runtime\n",
    f"- f_clue construction: {t_clue_elapsed:.1f}s",
    f"- f_common_wndef construction: {t_wndef_elapsed:.1f}s",
    f"- f_common_wnex construction: {t_wnex_elapsed:.1f}s",
    f"- Total notebook: {elapsed_total:.1f}s",
]

results_text = "\n".join(lines_out) + "\n"
results_path = OUTPUT_DIR / "results" / "02_phrase_construction_wn-results.md"
results_path.write_text(results_text)

print(results_text)
print(f"Results written to {results_path.relative_to(COMPONENT_ROOT)}")

---

## Summary

This notebook constructed phrase files for the three WordNet-based f
strategies and produced the associated vocabulary and filtered clue files.
These phrases are the raw text inputs to the embedding chain — the next
step (Stage 1d / Stage 4) passes them through CALE to produce the 1024-dim
embedding vectors used in triplet training and ATE evaluation.

**f_clue** wraps each definition in `<t></t>` delimiters within the clue
surface text, producing the "treatment" embedding that captures how
misleading clue context shifts the definition's perceived meaning. Rows where
`tag_definition_in_surface` returns `None` are excluded from `f_clue.csv`
but remain in the upstream `clues_wn_filtered.csv`.

**f_common_wndef** uses the most frequent WordNet synset's definition to
build `"<t>word</t>: definition"` phrases — the decontextualized baseline
for ATE comparison. Expected to cover all or nearly all vocabulary words,
since NB 01 already filtered to words with synsets.

**f_common_wnex** uses the most frequent synset's usage examples. Coverage
is substantially lower than wndef because many synsets lack examples and
some examples do not contain the target word exactly once. The wnex vocabulary
serves as the testbed for the formatting hypothesis.

**Output files (all under `data/filtered_split/wn_synset/`):**

| Directory | File | Description |
|-----------|------|-------------|
| `clue_phrases/` | `f_clue.csv` | Clue-contextualized phrases (treatment condition) |
| `wndef/` | `f_common_wndef.csv` | WordNet definition phrases (T=0 baseline) |
| `wndef/` | `vocabulary_wndef.csv` | Full wndef vocabulary |
| `wndef/` | `vocabulary_wndef_val.csv` | Validation-split wndef vocabulary |
| `wndef/` | `clues_wndef_filtered.csv` | Clue rows with both def and ans in wndef |
| `wnex/` | `f_common_wnex.csv` | WordNet usage example phrases (format test) |
| `wnex/` | `vocabulary_wnex.csv` | Full wnex vocabulary |
| `wnex/` | `vocabulary_wnex_val.csv` | Validation-split wnex vocabulary |
| `wnex/` | `clues_wnex_filtered.csv` | Clue rows with both def and ans in wnex |

**Results file:** `outputs/02_phrase_construction_wn-results.md`

See the results file for exact coverage statistics, split fractions,
cross-f comparisons, and runtimes.